In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [2]:
# Set seeds so you get the exact same random data every time
np.random.seed(42)
random.seed(42)

## Now i am creating a fake dummy data to practice my skills

In [10]:
# 1. Type 1 (b4, a23): Let's create 'Seat_Number'
seats = [random.choice(['A', 'B', 'C', 'D']) + str(random.randint(1, 99)) for _ in range(100)]

# 2. Type 2 (1, 2, a, d): Let's create 'Sensor_Reading' (mostly numbers, some text errors)
messy_readings = [random.randint(10, 100) if random.random() > 0.15 else random.choice(['a', 'd', 'error', '?']) for _ in range(100)]

# 3. Type 3 (Dates): Let's create 'Transaction_Date'
start_date = datetime(2026, 1, 1)
dates = [start_date + timedelta(days=random.randint(0, 180), hours=random.randint(0, 23)) for _ in range(100)]

# Create the DataFrame
df = pd.DataFrame({
    'Seat_Number': seats,
    'Sensor_Reading': messy_readings,
    'Transaction_Date': dates,
    'Target_Price': np.random.randint(100, 500, 100)
})

print("Here is our messy real-world data:")
display(df.head(6))
print("\nData Types:")
print(df.dtypes)

Here is our messy real-world data:


,Seat_Number,Sensor_Reading,Transaction_Date,Target_Price
0,A74,93,2026-01-18 14:00:00,484
1,A69,45,2026-03-14 09:00:00,371
2,B47,error,2026-03-25 02:00:00,288
3,C26,57,2026-05-22 14:00:00,291
4,D15,93,2026-01-03 11:00:00,168
5,C86,60,2026-02-21 09:00:00,377



Data Types:
Seat_Number                 object
Sensor_Reading              object
Transaction_Date    datetime64[ns]
Target_Price                 int64
dtype: object


# 1. i am fixing seat number and i will create two different columns in one i will store the letter and other the number this is a good way to store

In [21]:
# i use regex to remove alphabets
df['Seat_Letter'] = df['Seat_Number'].str.extract('([A-Za-z]+)')

# now i use regex to remove digits only
df['Seat_Num_Value'] = df['Seat_Number'].str.extract('([0-9]+)').astype(float)

df

,Seat_Number,Sensor_Reading,Transaction_Date,Target_Price,Seat_Letter,Seat_Num_Value
0,A74,93,2026-01-18 14:00:00,484,A,74.0
1,A69,45,2026-03-14 09:00:00,371,A,69.0
2,B47,error,2026-03-25 02:00:00,288,B,47.0
3,C26,57,2026-05-22 14:00:00,291,C,26.0
4,D15,93,2026-01-03 11:00:00,168,D,15.0
...,...,...,...,...,...,...
95,D3,19,2026-04-16 17:00:00,493,D,3.0
96,A21,25,2026-05-03 01:00:00,456,A,21.0
97,D59,16,2026-01-23 08:00:00,291,D,59.0
98,A55,76,2026-04-09 04:00:00,326,A,55.0


## 2. fixing the 1,2,a,d,error type the sensor_reading column

In [23]:
df['Sensor_Reading_Clean'] = pd.to_numeric(df['Sensor_Reading'], errors='coerce')
print("Now anything not a number is chaanged into nan")
display(df[['Sensor_Reading','Sensor_Reading_Clean']].head(10))

Now anything not a number is chaanged into nan


,Sensor_Reading,Sensor_Reading_Clean
0,93,93.0
1,45,45.0
2,error,NaN
3,57,57.0
4,93,93.0
5,60,60.0
6,99,99.0
7,64,64.0
8,100,100.0
9,d,NaN


## 3. Fixing type 3 date and time and how to deal with it

You cannot pass 2026-06-01 into a Random Forest. But, you can pass "Month: 6", "Day: 1", and "Is_Weekend: True". This is called Datetime Feature Extraction. Pandas has a built-in accessor called .dt that makes this incredibly easy.

In [25]:
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])

# Extract useful mathematical features
df['Month'] = df['Transaction_Date'].dt.month
df['Day'] = df['Transaction_Date'].dt.day
df['Hour'] = df['Transaction_Date'].dt.hour

# we can check things and create columns according to that
# e.g for weekends 1 else 0
df['IsWeekend'] = np.where(df['Transaction_Date'].dt.dayofweek > 4, 1, 0)

display(df.head())

,Seat_Number,Sensor_Reading,Transaction_Date,Target_Price,Seat_Letter,Seat_Num_Value,Sensor_Reading_Clean,Month,Day,Hour,IsWeekend
0,A74,93,2026-01-18 14:00:00,484,A,74.0,93.0,1,18,14,1
1,A69,45,2026-03-14 09:00:00,371,A,69.0,45.0,3,14,9,1
2,B47,error,2026-03-25 02:00:00,288,B,47.0,NaN,3,25,2,0
3,C26,57,2026-05-22 14:00:00,291,C,26.0,57.0,5,22,14,0
4,D15,93,2026-01-03 11:00:00,168,D,15.0,93.0,1,3,11,1


### Summary of Data Transformations (as of this point):

We have performed the following transformations to clean and extract features from the raw data:

1.  **Seat Number (`Seat_Number`)**: This column, originally a mix of letters and numbers (e.g., 'A74'), has been split into two new columns:
    *   `Seat_Letter`: Contains the alphabetic part (e.g., 'A').
    *   `Seat_Num_Value`: Contains the numeric part, converted to a float (e.g., 74.0).

2.  **Sensor Reading (`Sensor_Reading`)**: This column contained a mix of numeric values and textual errors (e.g., 'error', 'a', 'd'). It has been processed as follows:
    *   `Sensor_Reading_Clean`: All non-numeric values have been coerced to `NaN` (Not a Number), providing a clean numeric column for analysis.

3.  **Transaction Date (`Transaction_Date`)**: Useful temporal features have been extracted:
    *   `Month`: The month number.
    *   `Day`: The day of the month.
    *   `Hour`: The hour of the day.
    *   `IsWeekend`: A binary indicator (1 for weekend, 0 for weekday).

Let's display the DataFrame again to see all the new columns and the cleaned data.

In [26]:
print("Current state of the DataFrame after all transformations:")
display(df.head())

Current state of the DataFrame after all transformations:


,Seat_Number,Sensor_Reading,Transaction_Date,Target_Price,Seat_Letter,Seat_Num_Value,Sensor_Reading_Clean,Month,Day,Hour,IsWeekend
0,A74,93,2026-01-18 14:00:00,484,A,74.0,93.0,1,18,14,1
1,A69,45,2026-03-14 09:00:00,371,A,69.0,45.0,3,14,9,1
2,B47,error,2026-03-25 02:00:00,288,B,47.0,NaN,3,25,2,0
3,C26,57,2026-05-22 14:00:00,291,C,26.0,57.0,5,22,14,0
4,D15,93,2026-01-03 11:00:00,168,D,15.0,93.0,1,3,11,1
